# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a comprehensive, step-by-step guide for loading and exploring the [FAIR^2 Mass Spectrometry Extracellular Vesicles](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

All exploration will reference entities (record sets, fields, columns) using their `@id` attributes as defined in the Croissant schema.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # Metadata is an mlcroissant.Metadata object
print(f"{metadata.name}: {metadata.description}")
print(f"Schema version: {metadata.version}")
print(f"License: {metadata.license}")
print(f"Keywords: {metadata.keywords}")

## 2. Data Overview

Explore available record sets, fields, and their corresponding `@id` values.

Let's review the record sets and, for each one, summarize its schema (fields/columns and their IDs).

In [ ]:
# List available record sets
print("Available record sets (reporting their @id):")
record_sets = list(dataset.metadata.record_sets)
for rs in record_sets:
    print(f"@id: {rs.id}\n  name: {rs.name}\n  description: {rs.description}")
    print("  Fields/Columns:@ids:")
    for field in rs.fields:
        # Each field/column may have .id, .name, .data_type
        print(f"   - {field.id} (name: {field.name}, type: {field.data_type})")
    print()

## 3. Data Extraction

Load data from selected record sets into pandas DataFrames.

Below, we will demonstrate how to access each record set's data using its `@id`. Choose the desired record set for analysis.

In [ ]:
# Set up: collect all available record set @id's
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]
print("Record sets for extraction:", record_set_ids)

dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded {len(df)} records for RecordSet @id={record_set_id}")
    print(f"Columns: {df.columns.tolist()}")
    print("---")
# For further analysis, select one major table (assume the first if not specified)
main_record_set_id = record_set_ids[0] if record_set_ids else None
if main_record_set_id:
    print(f"\nFirst 5 rows of record set '@id': {main_record_set_id}")
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)

We'll demonstrate data processing with a numeric field from the selected record set. Adjust `numeric_field_id` and `group_field_id` as appropriate for your exploration (choose from the column names printed above, each mapping to a `Field` or `Column` `@id`).

In [ ]:
# Specify which record set to analyze
record_set_id = main_record_set_id  # Change if you want a different set
df = dataframes[record_set_id]
# Choose a numeric field and a group field (demo: pick the first float column if any)
numeric_field_id = None
group_field_id = None

# Heuristic: Detect first float or integer column by type or name
for col in df.columns:
    try:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_id = col
            break
    except Exception:
        continue
# Heuristic: pick any other column for grouping if available
for col in df.columns:
    if col != numeric_field_id:
        group_field_id = col
        break

print(f"Numeric field for analysis: {numeric_field_id}")
print(f"Group field for analysis: {group_field_id}")

# Filtering
if numeric_field_id and numeric_field_id in df.columns:
    threshold = df[numeric_field_id].mean() if not pd.isna(df[numeric_field_id].mean()) else 0
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records where {numeric_field_id} > {threshold:.2f} (mean value):")
    display(filtered_df.head())

    # Normalize the numeric field for filtered records
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Aggregation/grouping
    if group_field_id and group_field_id in filtered_df.columns:
        # Only numeric columns are aggregated, drop non-numeric for aggregate
        grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
        print(f"Grouped data by {group_field_id} (mean of numeric fields):")
        display(grouped_df.head())
else:
    print("No numeric field detected in the record set for analysis.")

## 5. Visualization

Visualize data distributions or relationships between fields in the dataset.

Below, we plot a histogram of the selected numeric field and a boxplot grouped by the group field (if any).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id and numeric_field_id in df.columns:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id], kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(10,5))
        sns.boxplot(data=df, x=group_field_id, y=numeric_field_id)
        plt.xticks(rotation=45)
        plt.title(f"{numeric_field_id} grouped by {group_field_id}")
        plt.show()
else:
    print("No numeric field available for visualization.")

## 6. Conclusion

In this notebook, we used the `mlcroissant` library to load, explore, and perform basic processing and visualization of the FAIR^2 dataset: *Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells*.

- All data operations referenced entities using their Croissant `@id`s for reproducibility and clarity.
- We reviewed record set and field structures, extracted records, filtered, normalized, and analyzed numeric data fields.
- Visualizations provide a first look at empirical distributions and patterns.

To further explore the dataset, select different fields and combine with domain knowledge for biological insights or downstream machine learning tasks.